# UERC26 Knowledge Distillation: TinyViT-21M → TinyViT-5M (A100 Optimized)
This notebook distills a pre-trained TinyViT-21M teacher into a TinyViT-11M student.
Optimized for A100 GPU with batch_size=256, bfloat16, and torch.compile.

**Teacher**: TinyViT-21M (576-dim features, loaded from Drive)
**Student**: TinyViT-5M (≈320-dim features, trained via KD)


In [ ]:
# 1. Mount Google Drive & Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install -q timm

!mkdir -p /content/checkpoints

zip_path = '/content/drive/MyDrive/UERC26_modified_new.zip'
import os
if not os.path.exists('/content/UERC26_modified_new/data/public/dataset_split.csv'):
    print('Unzipping dataset...')
    !unzip -q "{zip_path}" -d /content/
else:
    print('Dataset already unzipped.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset already unzipped.


In [ ]:
# 2. Import Modules
import os, csv, time, random, shutil
import numpy as np
from PIL import Image
import cv2
from pathlib import Path

cv2.setNumThreads(0)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.v2 as v2
import timm
from tqdm.notebook import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
# 3. CPU Preprocessing Utilities (CLAHE & Sharpening)
def apply_clahe(image_np, clip_limit=2.0, tile_grid_size=(8, 8)):
    lab = cv2.cvtColor(image_np, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def apply_sharpening(image_np, sigma=1.0, amount=0.5):
    blurred = cv2.GaussianBlur(image_np, (0, 0), sigma)
    sharpened = cv2.addWeighted(image_np, 1.0 + amount, blurred, -amount, 0)
    return np.clip(sharpened, 0, 255).astype(np.uint8)


In [ ]:
# 4. Optimized Dataset Classes
class EarIdentificationDataset(Dataset):
    """
    Optimized for lazy-loading WebP images on Colab.
    Applies CLAHE and Sharpening on CPU workers.
    """
    def __init__(self, split, root_dir, split_csv='dataset_split.csv',
                 classes_mapping=None, is_train=True):
        self.root_dir = root_dir
        self.split = split
        self.is_train = is_train

        self.images = []
        split_path = os.path.join(root_dir, split_csv)

        with open(split_path, 'r') as f:
            reader = csv.reader(f, delimiter='\t')
            for row in reader:
                if len(row) >= 2 and row[1] == split:
                    # Replace .jpg/.png with .webp
                    path_str = row[0].rsplit('.', 1)[0] + '.webp'
                    self.images.append(path_str)

        subjects = sorted(set(os.path.dirname(img) for img in self.images))
        if classes_mapping is not None:
            self.classes_mapping = classes_mapping
        else:
            self.classes_mapping = {s: i for i, s in enumerate(subjects)}
        self.num_classes = len(self.classes_mapping)

        # Filter to existing subjects
        self.images = [img for img in self.images if os.path.dirname(img) in self.classes_mapping]
        self.labels = [self.classes_mapping[os.path.dirname(img)] for img in self.images]

        self.transform = v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True)
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = str(Path(self.root_dir) / img_name)

        # Lazy loading strictly from disk
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Cannot read image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # CPU Augmentations (Only training)
        if self.is_train:
            if random.random() < 0.5:
                image = apply_clahe(image)
            if random.random() < 0.3:
                image = apply_sharpening(image)

        # To Tensor (Directly from numpy)
        image = self.transform(image)
        return image, self.labels[idx]

class EarPairwiseDataset(Dataset):
    """
    Evaluation Dataset. Assumes inputs are already 224x224.
    """
    def __init__(self, root_dir, pairs_csv):
        self.root_dir = root_dir
        self.pairs = []
        self.labels = []

        pairs_path = os.path.join(root_dir, pairs_csv)
        with open(pairs_path, 'r') as f:
            reader = csv.reader(f, delimiter='\t')
            for row in reader:
                if len(row) >= 3:
                    # Dynamic WebP replacement
                    p1 = row[0].rsplit('.', 1)[0] + '.webp'
                    p2 = row[1].rsplit('.', 1)[0] + '.webp'
                    self.pairs.append((p1, p2))
                    self.labels.append(int(row[2]))

        # Inputs are already 224x224. Just ToTensor and Normalize.
        self.transform = v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img1_path = str(Path(self.root_dir) / self.pairs[idx][0])
        img2_path = str(Path(self.root_dir) / self.pairs[idx][1])

        img1 = cv2.cvtColor(cv2.imread(img1_path), cv2.COLOR_BGR2RGB)
        img2 = cv2.cvtColor(cv2.imread(img2_path), cv2.COLOR_BGR2RGB)

        img1 = self.transform(img1)
        img2 = self.transform(img2)
        return img1, img2, self.labels[idx]


In [ ]:
# 5. Model Definitions: TinyViT-21M Teacher, TinyViT-11M Student & AdaFace

class TinyViT21MTeacher(nn.Module):
    """TinyViT-21M Teacher (576-dim backbone). Loaded from checkpoint, never trained."""
    def __init__(self, embedding_dim=512):
        super().__init__()
        self.backbone = timm.create_model('tiny_vit_21m_224.dist_in22k_ft_in1k', pretrained=True, num_classes=0)
        self.backbone_dim = 576
        self.projection = nn.Sequential(
            nn.Linear(self.backbone_dim, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
        )

    def forward(self, x, return_norms=False):
        features = self.backbone(x)
        embeddings = self.projection(features)
        norms = torch.norm(embeddings, p=2, dim=1)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return (embeddings, norms) if return_norms else embeddings


class TinyViT5MStudent(nn.Module):
    """TinyViT-5M Student (smaller backbone ~320-dim). Trained via Knowledge Distillation."""
    def __init__(self, embedding_dim=512):
        super().__init__()
        self.backbone = timm.create_model('tiny_vit_5m_224.dist_in22k_ft_in1k', pretrained=True, num_classes=0)
        self.backbone_dim = 320
        self.projection = nn.Sequential(
            nn.Linear(self.backbone_dim, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
        )

    def forward(self, x, return_norms=False):
        features = self.backbone(x)
        embeddings = self.projection(features)
        norms = torch.norm(embeddings, p=2, dim=1)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return (embeddings, norms) if return_norms else embeddings


class AdaFaceHead(nn.Module):
    def __init__(self, in_features, out_features, s=64.0, m=0.25, h=0.333, t_alpha=0.01):
        super().__init__()
        self.s, self.m, self.h, self.t_alpha = s, m, h, t_alpha
        self.out_features = out_features
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.register_buffer('batch_mean', torch.ones(1) * 20.0)
        self.register_buffer('batch_std', torch.ones(1) * 100.0)

    def forward(self, embeddings, norms, labels):
        with torch.no_grad():
            self.batch_mean = (1 - self.t_alpha) * self.batch_mean + self.t_alpha * norms.mean()
            self.batch_std  = (1 - self.t_alpha) * self.batch_std  + self.t_alpha * (norms.std() + 1e-6)
        margin_scaler = (norms - self.batch_mean) / (self.batch_std + 1e-3)
        margin_scaler = torch.clamp(self.h * margin_scaler, -1.0, 1.0)

        # Optimized: embeddings are already normalized by the model
        cos_theta = F.linear(embeddings, F.normalize(self.weight)).clamp(-1.0+1e-7, 1.0-1e-7)
        theta = torch.acos(cos_theta)
        one_hot = F.one_hot(labels, self.out_features).float()
        g_add = self.m * (1.0 + margin_scaler)
        target_logits = torch.cos(theta + self.m) - g_add.unsqueeze(1)
        output = (cos_theta * (1.0 - one_hot) + target_logits * one_hot) * self.s
        return F.cross_entropy(output, labels)


In [ ]:
# 6. Evaluation Function & Metrics
def compute_metrics(labels, scores):
    from scipy.interpolate import interp1d
    from sklearn.metrics import roc_curve, roc_auc_score
    fpr, tpr, _ = roc_curve(labels, scores)
    order = np.argsort(fpr)
    fpr, tpr = fpr[order], tpr[order]
    interp = interp1d(fpr, tpr, bounds_error=False, fill_value=(0.0, 1.0))
    metrics = {}
    metrics["VER@0.1%"] = float(interp(0.001))
    metrics["VER@1%"] = float(interp(0.01))
    fnr = 1 - tpr
    # Interpolated EER: find exact crossing of FPR == FNR
    abs_diff = np.abs(fnr - fpr)
    idx = np.argmin(abs_diff)
    if idx > 0 and abs_diff[idx - 1] < abs_diff[idx]:
        idx = idx - 1
    # Linear interpolation between the two closest points
    if idx < len(fpr) - 1:
        x0, x1 = fpr[idx], fpr[idx + 1]
        y0, y1 = fnr[idx], fnr[idx + 1]
        if (x1 - x0 + y0 - y1) != 0:
            eer = (x0 + y0) / 2  # fallback
            t = (y0 - x0) / ((x1 - x0) + (y0 - y1) + 1e-12)
            eer = x0 + t * (x1 - x0)
        else:
            eer = (fpr[idx] + fnr[idx]) / 2
    else:
        eer = (fpr[idx] + fnr[idx]) / 2
    metrics["EER"] = float(eer)
    metrics["AUC"] = float(roc_auc_score(labels, scores))
    return metrics

def evaluate_verification(model, dataloader, device):
    model.eval()
    all_scores, all_labels = [], []

    # Optimized: Inference mode + Autocast
    with torch.inference_mode(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        for img1, img2, labels in tqdm(dataloader, desc="Verif eval", leave=False):
            img1 = img1.to(device, non_blocking=True, memory_format=torch.channels_last)
            img2 = img2.to(device, non_blocking=True, memory_format=torch.channels_last)
            sim = F.cosine_similarity(model(img1), model(img2))
            all_scores.extend(sim.cpu().numpy())
            all_labels.extend(labels.numpy())

    return compute_metrics(np.array(all_labels), np.array(all_scores))


In [ ]:
# 7. Setup Dataloaders & Hardware (A100 Optimized)
DATA_ROOT = '/content/UERC26_modified_new/data/public'
BATCH_SIZE = 512   # A100: larger batch for smaller student; reduce if OOM
NUM_WORKERS = 12  # A100 host CPUs (increase if CPU cores available)
EPOCHS = 60
EVAL_INTERVAL = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')

print('Loading Datasets...')
train_dataset = EarIdentificationDataset('train', DATA_ROOT)
val_pair_dataset = EarPairwiseDataset(DATA_ROOT, 'pairs_val.csv') if os.path.exists(os.path.join(DATA_ROOT, 'pairs_val.csv')) else None

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    persistent_workers=True, prefetch_factor=4
)

if val_pair_dataset:
    val_loader = DataLoader(
        val_pair_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4
    )

print(f'Train samples: {len(train_dataset)}')

# GPU Transforms
gpu_transform = v2.Compose([
    v2.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    v2.RandomGrayscale(p=0.2),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


Loading Datasets...
Train samples: 179172


In [ ]:
# 8. Load Pre-Trained Teacher from Google Drive (NO training)
TEACHER_CHECKPOINT = '/content/drive/MyDrive/UERC26_Output_tinyvit_student/best_student_tinyvit.pt'

print('Loading TinyViT-21M Teacher from Google Drive...')
teacher_model = TinyViT21MTeacher(embedding_dim=512).to(device, memory_format=torch.channels_last)

teacher_ckpt = torch.load(TEACHER_CHECKPOINT, map_location=device)
teacher_model.load_state_dict(teacher_ckpt['model_state_dict'])

# Completely freeze teacher
for param in teacher_model.parameters():
    param.requires_grad = False

teacher_model.eval()

# Optimized: Compile the teacher for faster inference in the loop
teacher_model = torch.compile(teacher_model)

print(f'✅ Teacher loaded from {TEACHER_CHECKPOINT}')


Loading TinyViT-21M Teacher from Google Drive...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Teacher loaded from /content/drive/MyDrive/UERC26_Output_tinyvit_student/best_student_tinyvit.pt


In [ ]:
# 9. Setup Student Model, Optimizer & Scheduler
print('Initializing TinyViT-5M Student...')
student_model = TinyViT5MStudent(embedding_dim=512).to(device, memory_format=torch.channels_last)
student_loss_head = AdaFaceHead(in_features=512, out_features=train_dataset.num_classes).to(device)

# Compile for A100 speed (bfloat16 is native on Ampere)
student_model = torch.compile(student_model)

optimizer = torch.optim.AdamW([
    {'params': student_model.parameters(), 'lr': 1e-4},
    {'params': student_loss_head.parameters(), 'lr': 5e-4},
], weight_decay=1e-4, fused=True)

# Warmup + Cosine Annealing
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=2)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - 2, eta_min=1e-6)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup_scheduler, cosine_scheduler], milestones=[2])

mse_loss_fn = nn.MSELoss()
print('Setup complete! TinyViT-5M Student ready for distillation.')


Initializing TinyViT-5M Student...
Setup complete! TinyViT-5M Student ready for distillation.


In [ ]:
import torch

# 10. Dynamic Distillation Training Loop (bfloat16 for A100)
alpha_start = 0.90
alpha_end = 0.20

best_student_ver = 0.0
print('Starting Distillation: TinyViT-5M Student <- TinyViT-21M Teacher')

for epoch in range(1, EPOCHS + 1):
    student_model.train()
    student_loss_head.train()

    progress = (epoch - 1) / (EPOCHS - 1) if EPOCHS > 1 else 0
    current_alpha = alpha_start - progress * (alpha_start - alpha_end)

    total_loss, num_batches = 0.0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [Alpha: {current_alpha:.2f}]', leave=False)

    for images, labels in pbar:
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(device, non_blocking=True)
        images = gpu_transform(images)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            # Teacher pass (frozen, no grad)
            with torch.no_grad():
                teacher_embeddings = teacher_model(images).detach()

            # Student pass
            student_embeddings, student_norms = student_model(images, return_norms=True)

            # Loss
            loss_hard = student_loss_head(student_embeddings, student_norms, labels)
            loss_soft = mse_loss_fn(student_embeddings, teacher_embeddings)
            loss = (1.0 - current_alpha) * loss_hard + current_alpha * loss_soft

        # Backprop (no scaler needed for bfloat16)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student_model.parameters(), 1.0)
        torch.nn.utils.clip_grad_norm_(student_loss_head.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    print(f'Epoch {epoch} | Avg Loss: {total_loss/max(num_batches,1):.4f} | LR: {scheduler.get_last_lr()[0]:.2e}')

    # Evaluation
    if val_pair_dataset and epoch % EVAL_INTERVAL == 0:
        metrics = evaluate_verification(student_model, val_loader, device)
        print(f'STUDENT VAL | VER@0.1%: {metrics["VER@0.1%"]*100:.2f}% | EER: {metrics["EER"]*100:.2f}% | AUC: {metrics["AUC"]*100:.2f}%')

        if metrics['VER@0.1%'] > best_student_ver:
            best_student_ver = metrics['VER@0.1%']

            torch.save({
                'epoch': epoch,
                'model_state_dict': student_model.state_dict(),
                'loss_head_state_dict': student_loss_head.state_dict(),
                'val_auc': best_student_ver,
                'embedding_dim': 512,
                'backbone': 'tiny_vit_5m_224',
            }, '/content/checkpoints/best_student_tinyvit5m.pt')
            print('★ NEW BEST STUDENT MODEL SAVED!')

    scheduler.step()

print(f'Distillation Complete! Best Student VER@0.1: {best_student_ver*100:.2f}%')


Starting Distillation: TinyViT-5M Student <- TinyViT-21M Teacher


Epoch 1/60 [Alpha: 0.90]:   0%|          | 0/349 [00:00<?, ?it/s]

Epoch 1 | Avg Loss: 4.1596 | LR: 1.00e-05


Epoch 2/60 [Alpha: 0.89]:   0%|          | 0/349 [00:00<?, ?it/s]

Epoch 2 | Avg Loss: 4.1599 | LR: 5.50e-05


...

Epoch 44 | Avg Loss: 8.7532 | LR: 2.05e-05


Verif eval:   0%|          | 0/357 [00:00<?, ?it/s]

STUDENT VAL | VER@0.1%: 11.57% | EER: 15.65% | AUC: 91.58%


Epoch 45/60 [Alpha: 0.38]:   0%|          | 0/349 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# 11. Inference & Export to Google Drive
DRIVE_OUT_DIR = '/content/drive/MyDrive/2nd_time_UERC26_Output_TinyViT5M_Student_teacher_tinyvit21_superteacher_swinv2'
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

class EarImageDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.images = []
        for dirpath, _, filenames in os.walk(root_dir):
            for fname in sorted(filenames):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                    rel_path = os.path.relpath(os.path.join(dirpath, fname), root_dir)
                    self.images.append(rel_path.replace('\\', '/'))
        self.transform = v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img_name = self.images[idx]
        raw = cv2.imread(os.path.join(self.root_dir, img_name))
        if raw is None:
            raise FileNotFoundError(f'Cannot read: {os.path.join(self.root_dir, img_name)}')
        image = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
        image = apply_clahe(image)
        return self.transform(image), img_name

def extract_embeddings(model, dataloader, device):
    model.eval()
    features = {}
    # Optimized: Inference mode + Autocast
    with torch.inference_mode(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        for images, img_names in tqdm(dataloader, desc='Extracting', leave=False):
            emb = model(images.to(device)).cpu().numpy()
            for name, feat in zip(img_names, emb):
                features[name] = feat
    return features

print('Loading Best Student Model...')
ckpt = torch.load('/content/checkpoints/best_student_tinyvit5m.pt')
student_model.load_state_dict(ckpt['model_state_dict'])
student_model.eval()

seq_root = '/content/UERC26_modified_new/data/sequestered_anonymized'
local_out = '/content/checkpoints/tinyvit5m_solution'
os.makedirs(local_out, exist_ok=True)

if os.path.exists(seq_root):
    print(f'Extracting features for {seq_root}...')
    seq_dataset = EarImageDataset(seq_root)
    seq_loader = DataLoader(seq_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    seq_features = extract_embeddings(student_model, seq_loader, device)
    npy_path = os.path.join(local_out, 'sequestered_features.npy')
    np.save(npy_path, seq_features)
    print(f'Saved {len(seq_features)} embeddings.')
    shutil.copy(npy_path, DRIVE_OUT_DIR)
    shutil.copy('/content/checkpoints/best_student_tinyvit5m.pt', DRIVE_OUT_DIR)
    print(f'★ SUCCESS: Artifacts saved to {DRIVE_OUT_DIR}')
else:
    shutil.copy('/content/checkpoints/best_student_tinyvit5m.pt', DRIVE_OUT_DIR)
    print('★ Student model saved to Drive (no sequestered data found).')


In [ ]:
# 12. Final Student Evaluation
print('Loading Best Student for Final Evaluation...')
ckpt = torch.load('/content/checkpoints/best_student_tinyvit5m.pt')
student_model.load_state_dict(ckpt['model_state_dict'])
student_model.eval()

TEST_CSV = 'pairs_test.csv'
test_csv_path = os.path.join(DATA_ROOT, TEST_CSV)

if os.path.exists(test_csv_path):
    print(f'Evaluating on: {TEST_CSV}')
    test_pair_dataset = EarPairwiseDataset(DATA_ROOT, TEST_CSV)
    test_loader = DataLoader(
        test_pair_dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True, prefetch_factor=4
    )
    final_metrics = evaluate_verification(student_model, test_loader, device)
    print('\n' + '='*40)
    print('👑 FINAL STUDENT MODEL METRICS 👑')
    print('='*40)
    print(f'Backbone: TinyViT-5M (≈5M params)')
    print(f'Best Epoch: {ckpt["epoch"]}')
    print(f'VER @ 0.1% FAR : {final_metrics["VER@0.1%"]*100:.2f}%')
    print(f'VER @ 1.0% FAR : {final_metrics["VER@1%"]*100:.2f}%')
    print(f'Equal Error Rate : {final_metrics["EER"]*100:.2f}%')
    print(f'ROC AUC          : {final_metrics["AUC"]*100:.2f}%')
    print('='*40)
else:
    print(f'Could not find test dataset at: {test_csv_path}')


Loading Best Student for Final Evaluation...
Evaluating on: pairs_test.csv


Verif eval:   0%|          | 0/25 [00:00<?, ?it/s]


👑 FINAL STUDENT MODEL METRICS 👑
Backbone: TinyViT-5M (≈5M params)
Best Epoch: 14
VER @ 0.1% FAR : 44.09%
VER @ 1.0% FAR : 65.01%
Equal Error Rate : 10.60%
ROC AUC          : 95.38%


## A100 Tuning Suggestions
- Use `bfloat16` + `torch.compile()` for both teacher and student for speed.
- Use `channels_last` memory format and `pin_memory=True` in DataLoaders.
- Increase `BATCH_SIZE` (we set 512) but reduce if OOM; use gradient accumulation if needed.
- Use fused `AdamW` and `torch.optim.lr_scheduler.SequentialLR` (warmup + cosine) as configured.
- If you see performance regressions, try `torch.backends.cudnn.benchmark = True` and TF32 settings.
- For multi-GPU training: add DistributedDataParallel, sync batch-norm if needed.
- To debug OOM: enable `torch.cuda.empty_cache()` between epochs and lower `NUM_WORKERS` if CPU bound.
- Consider knowledge distillation variants: cosine embedding loss on normalized features, or temperature-scaled soft targets.
